Week 1 Task 2

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the dataset
df = pd.read_csv('/content/gapminder.csv')
display(df.head())
print(df.info())

### 1. The Messy Spaghetti Chart
We will plot life expectancy over time as one line per country (all 142) with a legend to see how hard it is to read.

In [ ]:
plt.figure(figsize=(12, 8))
sns.lineplot(data=df, x='year', y='lifeExp', hue='country', legend='full')
plt.title('Messy Chart: Life Expectancy over Time for All Countries')
plt.xlabel('Year')
plt.ylabel('Life Expectancy')
# Put legend outside because it has 142 items
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=4, fontsize='small')
plt.tight_layout()
plt.show()

### What is wrong with this messy chart?
1. **Overplotting (Spaghetti Effect):** With 142 overlapping lines, individual trends are impossible to trace.
2. **Legend Bloat:** A legend with 142 entries is completely unreadable, takes up more space than the plot itself, and many colors are indistinguishably repeated.
3. **Visual Noise:** The visual density is so high that it hides any meaningful story or outliers.

### Fix (a): Aggregate — One line per continent

In [ ]:
# Calculate mean life expectancy per continent per year
continent_df = df.groupby(['continent', 'year'])['lifeExp'].mean().reset_index()

plt.figure(figsize=(10, 6))
sns.lineplot(data=continent_df, x='year', y='lifeExp', hue='continent', marker='o', linewidth=2.5)
plt.title('Fix (a): Average Life Expectancy over Time by Continent')
plt.xlabel('Year')
plt.ylabel('Average Life Expectancy')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### Fix (b): Highlight — Grey out background, color 3 interesting countries
Let's highlight three interesting countries: **Japan** (extraordinary growth and top rank), **Rwanda** (showing the dramatic dip during the 1994 genocide), and **Zimbabwe** (showing the severe impact of the HIV/AIDS epidemic in the 1990s and subsequent recovery).

In [ ]:
plt.figure(figsize=(12, 8))

# Plot all countries in light grey
for country in df['country'].unique():
    country_data = df[df['country'] == country]
    plt.plot(country_data['year'], country_data['lifeExp'], color='lightgrey', alpha=0.5, linewidth=1)

# Highlight 3 specific countries
highlights = {
    'Japan': 'crimson',
    'Rwanda': 'royalblue',
    'Zimbabwe': 'forestgreen'
}

for country, color in highlights.items():
    country_data = df[df['country'] == country]
    plt.plot(country_data['year'], country_data['lifeExp'], color=color, linewidth=3, label=country)
    # Label directly on the chart (near the last data point)
    last_row = country_data.iloc[-1]
    plt.text(last_row['year'] + 0.5, last_row['lifeExp'], country, color=color, fontweight='bold', va='center')

plt.title('Fix (b): Highlighted Countries vs. Global Trends')
plt.xlabel('Year')
plt.ylabel('Life Expectancy')
plt.xlim(1950, 2012) # Extend x-axis slightly for labels
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

### Comparing (a) and (b):
- **What (a) Aggregate shows:** It clearly highlights macro-level continental trends, revealing that overall life expectancy has increased across all continents, while preserving the relative rankings of continental averages (e.g., Africa remaining lower than Europe/Americas).
- **What (b) Highlight shows:** It preserves the full visual variance of the dataset in the background (showing the overall spread and envelope of human progress) while focusing attention on specific national narratives and historical anomalies (like Rwanda's crisis or Japan's elite trajectory) that are completely smoothed away in a continent-wide average.

### GDP Skewness: Linear vs. Log Scale
Let's first plot the histogram of `gdpPercap` to see the skewness, and then see how a log scale affects it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear Scale Histogram
sns.histplot(df['gdpPercap'], bins=30, ax=axes[0], kde=True, color='teal')
axes[0].set_title('GDP per Capita Distribution (Linear Scale)')
axes[0].set_xlabel('GDP per Capita')

# Log Scale Histogram
sns.histplot(df['gdpPercap'], bins=30, ax=axes[1], kde=True, log_scale=True, color='orange')
axes[1].set_title('GDP per Capita Distribution (Log Scale)')
axes[1].set_xlabel('GDP per Capita (Log Scale)')

plt.tight_layout()
plt.show()

### Analysis of the GDP Scale:
- **Linear Scale:** Shows an extremely right-skewed distribution where almost all countries sit in a low-income bin, making it very hard to differentiate between low, middle, and high income levels.
- **Log Scale:** Normalizes the distribution, spreading out the observations and letting us easily compare relative ratios (e.g., doubling of income is represented by equal distance on the plot).

### Has the gap between the richest and poorest narrowed since 1952?
We will construct a chart showing the trend of the gap using the ratio between the 90th percentile (representing the richest) and the 10th percentile (poorest) over time.

*Justification for using the 90/10 Ratio:* Using absolute dollar differences is misleading because inflation and overall global growth expand absolute gaps, even if relative inequality shrinks. A ratio metric demonstrates relative economic disparity over time.

In [ ]:
# Calculate 10th and 90th percentiles of GDP per Capita for each year
gap_df = df.groupby('year')['gdpPercap'].agg(
    p10=lambda x: np.percentile(x, 10),
    p90=lambda x: np.percentile(x, 90)
).reset_index()

gap_df['gdp_ratio_90_10'] = gap_df['p90'] / gap_df['p10']

plt.figure(figsize=(10, 6))
sns.lineplot(data=gap_df, x='year', y='gdp_ratio_90_10', marker='s', color='darkred', linewidth=2.5)
plt.title('Disparity Ratio: 90th Percentile / 10th Percentile GDP per Capita over Time')
plt.xlabel('Year')
plt.ylabel('GDP per Capita Ratio (Richest / Poorest)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### Findings:
As shown by the 90/10 ratio chart, the relative gap between the richest 10% and the poorest 10% of countries actually **widened significantly** from 1952 until around 1997, after which it began to level off and slightly decline. It remains much higher today than it was in 1952.